# What's New in v0.4.0

> **Note:** This notebook uses v0.4.0. Run each cell in order; set the required environment variables before starting.

A guided tour of all breaking changes and new features added in v0.4.0.

**Sections:**
1. Setup
2. Breaking changes from v0.3.x
3. New block types
4. Markdown Content API
5. Markdown comments
6. Views API
7. New filter and property features
8. Native icons and custom emojis

## 1. Setup

In [6]:
@file:DependsOn("it.saabel:kotlin-notion-client:0.4.1")

import it.saabel.kotlinnotionclient.NotionClient
import it.saabel.kotlinnotionclient.models.base.*
import it.saabel.kotlinnotionclient.models.pages.*
import it.saabel.kotlinnotionclient.models.databases.*
import it.saabel.kotlinnotionclient.models.blocks.*
import it.saabel.kotlinnotionclient.models.datasources.*
import it.saabel.kotlinnotionclient.models.views.*
import kotlinx.coroutines.runBlocking

val apiToken = System.getenv("NOTION_API_TOKEN")
    ?: error("NOTION_API_TOKEN not set")
val parentPageId = System.getenv("NOTION_TEST_PAGE_ID")
    ?: error("NOTION_TEST_PAGE_ID not set")

val notion = NotionClient(apiToken)

// Create a self-contained demo database used by the Views and Filter sections.
val demoDb = runBlocking {
    notion.databases.create {
        parent.page(parentPageId)
        title("v0.4 Demo Database")
        icon.emoji("🔬")
        properties {
            title("Name")
            date("Due Date")
            people("Assignee")
        }
    }
}
val dataSourceId = demoDb.dataSources.first().id

// Create a demo page used by other sections.
val demoPage = runBlocking {
    notion.pages.create {
        parent.page(parentPageId)
        title("Breaking Changes Demo")
        icon.emoji("🔧")
        content { paragraph("Demo page for v0.4 breaking changes.") }
    }
}
println("Client initialized")
println("Created demo page: ${demoPage.url}")
println("Demo database: ${demoDb.url}")

Client initialized
Created demo page: https://www.notion.so/Breaking-Changes-Demo-343c63fd82ed81e3a2d1f6c8352d7c71
Demo database: https://www.notion.so/3c1c5d7b203c4668a9569354e333e719


## 2. Breaking Changes from v0.3.x

### 2.1 `inTrash` replaces `archived`

The `archived` field is now `inTrash` (`in_trash` in the API). All page, database, block, and comment objects use `inTrash`.

In [7]:
println("inTrash: ${demoPage.inTrash}") // false — not in trash

inTrash: false


### 2.2 `trash()` replaces `archive()`

The `archive()` / `unarchive()` methods have been renamed to match Notion's official terminology.

In [8]:
import kotlinx.coroutines.delay

// Trash the page
val trashed = runBlocking { notion.pages.trash(demoPage.id) }
println("After trash()  → inTrash: ${trashed.inTrash}")  // true

// Restore it
val restored = runBlocking {
    delay(3.seconds)
    notion.pages.update(demoPage.id) { trash(false) }
}
println("After restore() → inTrash: ${restored.inTrash}") // false

After trash()  → inTrash: true
After restore() → inTrash: false


### 2.3 `BlockAppendPosition` typed sealed class

The `after: String?` parameter was replaced by `position: BlockAppendPosition?`.

In [9]:
// Append a block at the default position (end)
val appended = runBlocking {
    notion.blocks.appendChildren(demoPage.id) {
        paragraph("Block A — appended at end (default)")
        paragraph("Block B — appended at end (default)")
    }
}
val blockA = appended.results[0]
val blockB = appended.results[1]

// Prepend a block at the start of the page
runBlocking {
    notion.blocks.appendChildren(demoPage.id, position = BlockAppendPosition.Start) {
        paragraph("Block C — prepended at start")
    }
}

// Insert a block after Block A specifically
runBlocking {
    notion.blocks.appendChildren(demoPage.id, position = BlockAppendPosition.AfterBlock(BlockReference(blockA.id))) {
        paragraph("Block D — inserted after Block A")
    }
}

println("Check ${demoPage.url} — order should be: C, A, D, B")

Check https://www.notion.so/Breaking-Changes-Demo-343c63fd82ed81e3a2d1f6c8352d7c71 — order should be: C, A, D, B


### 2.4 Unified `Icon` sealed class

`PageIcon` and `CalloutIcon` are gone. All icon fields now use the unified `Icon` sealed class. `Icon.NativeIcon` is new in v0.4.0.

In [10]:
// Emoji icon (was PageIcon.Emoji)
val withEmoji = runBlocking {
    notion.pages.update(demoPage.id) { icon.emoji("🚀") }
}
println("Emoji icon:  ${(withEmoji.icon as? Icon.Emoji)?.emoji}")

// Native icon — new in v0.4.0
val withNative = runBlocking {
    notion.pages.update(demoPage.id) { icon.native("star", NativeIconColor.YELLOW) }
}
println("Native icon: ${(withNative.icon as? Icon.NativeIcon)?.icon?.name} / ${(withNative.icon as? Icon.NativeIcon)?.icon?.color}")

// External URL icon (was PageIcon.External)
val withExternal = runBlocking {
    notion.pages.update(demoPage.id) {
        icon.external("https://www.notion.so/icons/target_gray.svg")
    }
}
println("External icon: ${(withExternal.icon as? Icon.External)?.external?.url}")

Emoji icon:  🚀
Native icon: star / YELLOW
External icon: null


## 3. New Block Types

### 3.1 `heading_4`

In [11]:
val blockResult = runBlocking {
    notion.blocks.appendChildren(demoPage.id) {
        heading1("Architecture")
        heading2("Frontend")
        heading3("Components")
        heading4("Atoms")  // New in v0.4.0
        paragraph("The smallest reusable elements.")
    }
}
println("Appended ${blockResult.results.size} blocks")

Appended 5 blocks


### 3.2 `tab` block

Container for tab-pane layouts:

In [12]:
val tabResult = runBlocking {
    notion.blocks.appendChildren(demoPage.id) {
        tab {
            pane("Overview") {
                paragraph("Project summary.")
            }
            pane("Details", icon = emoji("🎯")) {
                bullet("Key detail one")
            }
            pane("Settings", icon = nativeIcon("gear", NativeIconColor.GRAY)) {
            }
        }
    }
}
println("Tab block: ${tabResult.results.first().id}")

Tab block: 343c63fd-82ed-817a-97fb-dc4dde5c71df


### 3.3 `meeting_notes` (read-only)

Meeting notes blocks are returned by the API but cannot be created programmatically.

## 4. Markdown Content API

`client.markdown` — retrieve, create, and update page content as Markdown text.

### 4.1 Retrieve page as markdown

In [13]:
val mdResponse = runBlocking { notion.markdown.retrieve(demoPage.id) }
println("Truncated: ${mdResponse.truncated}")
println(mdResponse.markdown.take(300))

Truncated: false
Block C — prepended at start
Demo page for v0.4 breaking changes.
Block A — appended at end (default)
Block D — inserted after Block A
Block B — appended at end (default)
# Architecture
## Frontend
### Components
#### Atoms
The smallest reusable elements.
<tabs>
	<tab>
		Overview
		Project summary.



### 4.2 Create a page with markdown

In [14]:
val mdPage = runBlocking {
    notion.pages.create {
        parent.page(demoPage.id)
        icon.emoji("📝")
        title("Markdown Test Page")
        markdown(
            "# Hello\n\nCreated with the **Markdown Content API**.\n\n- Item one\n- Item two"
        )
    }
}
println("Created: ${mdPage.id}")

Created: 343c63fd-82ed-8166-9a06-ccb650032982


### 4.3 Search-and-replace

In [15]:
val updated = runBlocking {
    notion.markdown.updateContent(mdPage.id) {
        replace("Item one", "Updated item one")
    }
}
println(updated.markdown.take(300))

Created with the **Markdown Content API**.
- Updated item one
- Item two


### 4.4 Full page replace

In [16]:
runBlocking {
    notion.markdown.replaceContent(mdPage.id, "# Replaced\n\nContent replaced.")
}
println("Replaced!")

Replaced!


## 5. Markdown Comments

`markdown()` is an alternative to `richText {}` in comment builders:

In [17]:
val comment = runBlocking {
    notion.comments.create {
        parent.page(mdPage.id)
        markdown("Comment in **Markdown** with `code` and _italic_.")
    }
}
println("Comment: ${comment.id}")

Comment: 343c63fd-82ed-81fd-b17d-001d659580a4


## 6. Views API

`client.views` — full CRUD for database views.

### 6.1 List views

In [18]:
// Seed demoDb with a few pages so the view has real data to display
runBlocking {
    listOf(
        Triple("Design system audit", "2026-05-01", false),
        Triple("API rate limit review", "2026-05-10", false),
        Triple("Onboarding docs", "2026-04-20", true),
    ).forEach { (name, due, _) ->
        notion.pages.create {
            parent.dataSource(dataSourceId)
            properties {
                title("Name", name)
                date("Due Date", due)
            }
        }
    }
}
println("Seeded 3 pages into demoDb")

Seeded 3 pages into demoDb


In [19]:
// listAsFlow emits ViewReference (id only) — retrieve for full details
val viewRefs = mutableListOf<ViewReference>()
runBlocking {
    notion.views.listAsFlow(dataSourceId = dataSourceId).collect { v: ViewReference ->
        viewRefs += v
    }
}
println("Found ${viewRefs.size} view(s)")

// Retrieve the first one to inspect name and type
val firstView = runBlocking { notion.views.retrieve(viewRefs.first().id) }
println("First view — name: ${firstView.name}, type: ${firstView.type}")

Found 1 view(s)
First view — name: Default view, type: TABLE


### 6.2 Create a table view

In [20]:
val tableView = runBlocking {
    notion.views.create {
        database(demoDb.id)
        dataSourceId(dataSourceId)
        type(ViewType.TABLE)
        name("v0.4 Table View")
    }
}
println("Table view: ${tableView.id}")

Table view: 343c63fd-82ed-813a-a86d-000c004176ad


### 6.3 Typed ViewConfiguration (gallery)

In [21]:
val galleryView = runBlocking {
    notion.views.create {
        database(demoDb.id)
        dataSourceId(dataSourceId)
        type(ViewType.GALLERY)
        name("v0.4 Gallery View")
        configuration(
            ViewConfiguration.Gallery(
                coverSize = CoverSize.MEDIUM,
            )
        )
    }
}
println("Gallery view: ${galleryView.id}")

Gallery view: 343c63fd-82ed-8160-8b04-000cf8f74b5e


### 6.4 Update a view, control property visibility, and delete

In [22]:
// Rename the table view
val renamed = runBlocking {
    notion.views.update(tableView.id) {
        name("Task Table (renamed)")
    }
}
println("Renamed to: ${renamed.name}")

// Retrieve data source schema to get property IDs
val schema = runBlocking { notion.dataSources.retrieve(dataSourceId) }
val propIds = schema.properties.mapValues { it.value.id }
println("Property IDs: $propIds")

// On the gallery view: show only Name, hide Due Date and Assignee
val dueDateId = propIds["Due Date"] ?: error("Due Date property not found")
val assigneeId = propIds["Assignee"] ?: error("Assignee property not found")

val slimGallery = runBlocking {
    notion.views.update(galleryView.id) {
        type(ViewType.GALLERY)
        hideProperties(dueDateId, assigneeId)
    }
}
val hidden = slimGallery.configuration
    ?.let { it as? ViewConfiguration.Gallery }
    ?.properties?.filter { it.visible == false }
    ?.map { it.propertyId } ?: emptyList()
println("Hidden property IDs: $hidden")

// Delete the table view — its story is complete (created → renamed → deleted)
// The gallery view stays alive so you can inspect its property config in Notion
runBlocking { notion.views.delete(tableView.id) }
println("Table view deleted. Open ${galleryView.url} to see the gallery with hidden properties.")

Renamed to: Task Table (renamed)
Property IDs: {Assignee=VOwg, Due Date=%5DZ%7Dt, Name=title}
Hidden property IDs: []Z}t, VOwg, title]
Table view deleted. Open https://www.notion.so/3c1c5d7b203c4668a9569354e333e719?v=343c63fd82ed81608b04000cf8f74b5e to see the gallery with hidden properties.


## 7. New Filter and Property Features

### 7.1 Relative date filters

In [23]:
import it.saabel.kotlinnotionclient.models.datasources.RelativeDateValue

val upcoming = runBlocking {
    notion.dataSources.query(dataSourceId) {
        filter {
            date("Due Date").onOrAfter(RelativeDateValue.TODAY)
            date("Due Date").onOrBefore(RelativeDateValue.ONE_WEEK_FROM_NOW)
        }
    }
}
println("Due this week: ${upcoming.size}")

Due this week: 1


### 7.2 People `containsMe()` filter

> **Note:** Internal integrations always return 0 / all results for `containsMe()` / `doesNotContainMe()`. These filters are only meaningful for user-level (OAuth) integrations, where "me" resolves to the authenticated user.

In [24]:
val myItems = runBlocking {
    notion.dataSources.query(dataSourceId) {
        filter {
            people("Assignee").containsMe()
        }
    }
}
println("My items: ${myItems.size}")

My items: 0


### 7.3 Status property in database creation

In [26]:
val sprintDb = runBlocking {
    notion.databases.create {
        parent.page(demoPage.id)
        title("Sprint Board (v0.4 demo)")
        properties {
            title("Task")
            status("State") {
                option("Backlog", SelectOptionColor.GRAY)
                option("In Progress", SelectOptionColor.YELLOW)
                option("Done", SelectOptionColor.GREEN)
            }
        }
    }
}
println("Sprint board: ${sprintDb.id}, ${sprintDb.url}")


Sprint board: b58b802c-f8d7-49a4-b2fc-804797abb3d0, https://www.notion.so/b58b802cf8d749a4b2fc804797abb3d0


## 8. Native Icons and Custom Emojis

### 8.1 Native icon on a page

In [27]:
val iconPage = runBlocking {
    notion.pages.update(demoPage.id) {
        icon.native("star", NativeIconColor.YELLOW)
    }
}
println("Icon: ${iconPage.icon}")

Icon: NativeIcon(type=icon, icon=NativeIconObject(name=star, color=YELLOW))


### 8.2 Native icon in callout blocks

In [28]:
runBlocking {
    notion.blocks.appendChildren(demoPage.id) {
        callout(nativeIcon("gear", NativeIconColor.GRAY)) {
            text("Settings note")
        }
    }
}
println("Callout with native icon created")

Callout with native icon created


### 8.3 List custom emojis

In [29]:
runBlocking {
    var count = 0
    notion.customEmojis.listAsFlow().collect { emoji ->
        if (count < 3) println("${emoji.name}: ${emoji.id}")
        count++
    }
    if (count == 0) println("No custom emojis in this workspace")
    else println("Total: $count custom emoji(s)")
}

faf25-profile-pic-1: 219c63fd-82ed-804a-8144-007a62154105
faf25-profile-pic: 219c63fd-82ed-807f-98a2-007aaa83dc20
eventive: 247c63fd-82ed-805a-bf81-007ae9f866f5
Total: 3 custom emoji(s)


## Cleanup

Trash all resources created during this notebook:

- **Demo database** (`demoDb`) — lives alongside the demo page under `NOTION_TEST_PAGE_ID`; trashed explicitly.
- **Demo page** (`demoPage`) — the main working page for sections 2–5 and 7–8. Trashing it cascades to the **sprint board** (`sprintDb`) and the **markdown test page** (`mdPage`), which are both children of `demoPage`.

In [30]:
runBlocking {
    notion.databases.trash(demoDb.id)   // demo database + its views and seeded pages
    notion.pages.trash(demoPage.id)     // demo page + sprintDb + mdPage (cascade)
}
notion.close()
println("✅ All resources trashed — notebook complete!")

✅ All resources trashed — notebook complete!
